# Stim split-stability notebook for LOCAT 0.1

This notebook mirrors the original preprocessing setup, then focuses on
`stim` cells only and asks three related questions:

- **Split stability**: if we rerun the full pipeline independently on two
  disjoint `stim` subsets, do we recover similar localized genes?
- **PC sensitivity**: within each fold, how much do LOCAT rankings change
  across `4`, `8`, and `12` PCA dimensions?
- **Seed sensitivity**: within each fold and PC setting, how variable are
  results across repeated LOCAT runs?

Design:
- build **two disjoint folds**, each containing about **1/3** of stim cells
- leave the remaining ~1/3 unused
- define one shared **2000-gene** panel used everywhere
- for each fold, run LOCAT at `4`, `8`, and `12` PCs
- for each fold/PC setting, run LOCAT with `3` fixed seeds


In [ ]:
!nvidia-smi


In [ ]:
import os
import sys

sys.path.insert(0, 'LOCAT01_PATH')

for mod in list(sys.modules):
    if mod == 'locat' or mod.startswith('locat.'):
        del sys.modules[mod]

os.environ['CUDA_VISIBLE_DEVICES'] = os.environ.get('CUDA_VISIBLE_DEVICES', '1')


In [ ]:
import numpy as np
from numpy import random

SEED = 13

def reset_seeds(seed=SEED):
    np.random.seed(seed)
    random.seed(seed)
    return seed

reset_seeds(SEED)


In [ ]:
import re
import json
import scipy.sparse as sp
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import seaborn as sns

from itertools import combinations
from pathlib import Path
from matplotlib import pyplot as plt
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm

from locat.locat import LOCAT

sc.settings.set_figure_params(dpi=110, facecolor='white')
sns.set_context('notebook')


In [ ]:
adata = sc.read(
    "../../../data/kang_counts_25k.h5ad",
    backup_url="https://figshare.com/ndownloader/files/34464122",
)
adata


In [ ]:
adata.layers["counts"] = adata.X.copy()


In [ ]:
if "counts" in adata.layers:
    counts_layer = "counts"
else:
    counts_layer = None

if counts_layer is None:
    adata.layers["counts"] = adata.X.copy()
    counts_layer = "counts"

sc.pp.normalize_total(adata, target_sum=1e4, layer=counts_layer)
sc.pp.log1p(adata, layer=counts_layer)

adata.X = adata.layers[counts_layer].copy()
adata.raw = adata.copy()


In [ ]:
pseudo_patterns = [
    r"^Gm\d+",
    r"Rik$",
    r"^AC\d+",
    r"^AA\d+",
    r"^A[0-9]{6,}",
    r"^Mir\d+",
    r"^Rpl\d*-\d+",
    r"^Rps\d*-\d+",
    r"^Linc",
]

pseudo_re = re.compile("|".join(f"(?:{p})" for p in pseudo_patterns))
gene_series = pd.Index(adata.var_names.astype(str))
is_pseudo = gene_series.to_series().str.match(pseudo_re)

print("Genes flagged as pseudo-ish:", int(is_pseudo.sum()), "of", adata.n_vars)
adata = adata[:, ~is_pseudo.values].copy()
print("After filtering:", adata.shape)


In [ ]:
counts = adata.layers["counts"]
n_cells = adata.n_obs

if sp.issparse(counts):
    n_cells_expressed = np.array((counts > 0).sum(axis=0)).ravel()
else:
    n_cells_expressed = (counts > 0).sum(axis=0)

frac_expressed = n_cells_expressed / n_cells
keep = frac_expressed > 0.01

print(f"Keeping {keep.sum()} genes out of {adata.n_vars}")
adata = adata[:, keep].copy()


In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)


In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)


In [ ]:
sc.pl.umap(
    adata,
    color=["label", "cell_type"],
    frameon=False,
    ncols=2,
)


## Stim subset and helper functions


In [ ]:
stim = adata[adata.obs["label"] == "stim"].copy()
stim


In [ ]:
print(stim.obs["cell_type"].value_counts())


In [ ]:

PCS_LIST = [4, 8, 12]
BIPCA_SETTING = "bipca_umap2d"
SETTING_LIST = PCS_LIST + [BIPCA_SETTING]
STANDARD_UMAP_PCS = [50, 4, 8, 12]
LOCAT_SEEDS = [13, 113, 1013]
SEED_LABELS = [f"seed_{seed}" for seed in LOCAT_SEEDS]
SETTING_LABELS = {
    4: "4 PCs",
    8: "8 PCs",
    12: "12 PCs",
    BIPCA_SETTING: "BiPCA rank -> UMAP2D",
}
SETTING_SHORT_LABELS = {
    4: "4",
    8: "8",
    12: "12",
    BIPCA_SETTING: "BiPCA->UMAP2D",
}
from bipca import BiPCA
from datetime import datetime
import time

PROGRESS_LOG = Path('Perturb_PBMC_stim_cv_twothirds_progress.log')
INTERMEDIATE_DIR = Path('support_files/stim_twothirds_cv_intermediate')
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(exist_ok=True, parents=True)
PROGRESS_LOG.write_text(f"[{datetime.now().isoformat(timespec='seconds')}] Starting notebook run\n")


def log_progress(message):
    line = f"[{datetime.now().isoformat(timespec='seconds')}] {message}"
    print(line, flush=True)
    with open(PROGRESS_LOG, 'a', encoding='utf-8') as fh:
        fh.write(line + "\n")


def setting_slug(setting):
    return f"pcs{setting}" if isinstance(setting, int) else str(setting)


def setting_label(setting):
    return SETTING_LABELS[setting]


def setting_short_label(setting):
    return SETTING_SHORT_LABELS[setting]


def setting_embedding_key(setting):
    if isinstance(setting, int):
        return f"X_umap_pcs{setting}"
    if setting == BIPCA_SETTING:
        return "X_umap_bipca_umap2d"
    raise ValueError(f"Unknown setting: {setting}")


def to_dense_1d(x):
    if sparse.issparse(x):
        x = x.toarray()
    return np.asarray(x).ravel()


def ensure_dense_X(adata_in):
    if sparse.issparse(adata_in.X):
        adata_in.X = np.asarray(adata_in.X.toarray(), dtype=np.float64)
    else:
        adata_in.X = np.asarray(adata_in.X, dtype=np.float64)
    return adata_in


def get_strat_labels(obs_df):
    counts = obs_df["cell_type"].value_counts()
    rare = counts[counts < 2].index
    labels = obs_df["cell_type"].astype(str).copy()
    if len(rare) > 0:
        labels.loc[labels.isin(rare)] = "OTHER_RARE"
    return labels


def build_two_disjoint_folds(stim_adata, frac=1/3, random_state=SEED):
    labels = get_strat_labels(stim_adata.obs)
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=frac, random_state=random_state)
    all_idx = np.arange(stim_adata.n_obs)
    _, fold1_idx = next(splitter1.split(np.zeros(stim_adata.n_obs), labels))

    remaining_mask = np.ones(stim_adata.n_obs, dtype=bool)
    remaining_mask[fold1_idx] = False
    rem_idx = all_idx[remaining_mask]
    rem_labels = labels.iloc[rem_idx]

    target_frac_of_remaining = frac / (1 - frac)
    splitter2 = StratifiedShuffleSplit(
        n_splits=1,
        test_size=target_frac_of_remaining,
        random_state=random_state + 1,
    )
    _, fold2_subidx = next(splitter2.split(np.zeros(len(rem_idx)), rem_labels))
    fold2_idx = rem_idx[fold2_subidx]

    unused_mask = np.ones(stim_adata.n_obs, dtype=bool)
    unused_mask[fold1_idx] = False
    unused_mask[fold2_idx] = False
    unused_idx = all_idx[unused_mask]

    fold_dict = {
        "fold_A": stim_adata[fold1_idx].copy(),
        "fold_B": stim_adata[fold2_idx].copy(),
        "unused_holdout": stim_adata[unused_idx].copy(),
    }
    return fold_dict


def counts_stats_for_fold(fold):
    counts = fold.layers["counts"]
    if sparse.issparse(counts):
        expr_frac = np.asarray((counts > 0).mean(axis=0)).ravel()
    else:
        expr_frac = (counts > 0).mean(axis=0)

    X = fold.X
    if sparse.issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float64)
    variance = X.var(axis=0, ddof=1)
    return expr_frac, variance


def select_shared_genes(fold_dict, n_genes=2000, min_frac=0.01, max_frac=0.80):
    use_folds = ["fold_A", "fold_B"]
    all_var_names = fold_dict[use_folds[0]].var_names
    eligible = np.ones(len(all_var_names), dtype=bool)
    variances = []

    for fold_name in use_folds:
        expr_frac, variance = counts_stats_for_fold(fold_dict[fold_name])
        eligible &= (expr_frac > min_frac) & (expr_frac < max_frac)
        variances.append(variance)

    avg_variance = np.mean(np.vstack(variances), axis=0)
    selected = (
        pd.DataFrame(
            {
                "gene": all_var_names,
                "eligible": eligible,
                "avg_variance": avg_variance,
            }
        )
        .query("eligible")
        .sort_values("avg_variance", ascending=False)
        .head(n_genes)
    )
    return selected.reset_index(drop=True)


def compute_bipca_rank(fold, seed=SEED):
    X_counts = fold.layers.get("counts", fold.X)
    op = BiPCA(
        n_components=-1,
        seed=seed,
        qits=11,
        n_subsamples=2,
        exact=False,
        subsample_size=2000,
        backend="scipy",
        svd_backend="scipy",
        sinkhorn_backend="scipy",
        verbose=0,
        suppress=True,
        njobs=1,
    )
    op.fit(X_counts)
    rank = getattr(op, "mp_rank", None)
    if rank is None:
        raise ValueError("BiPCA fit did not produce mp_rank")
    rank = int(rank)
    rank = max(2, min(rank, fold.obsm["X_pca"].shape[1]))
    return rank


def prepare_fold(fold, genes_df, fold_name):
    fold_start = time.time()
    genes = genes_df["gene"].tolist()
    fold = fold[:, genes].copy()
    ensure_dense_X(fold)
    log_progress(f"{fold_name}: start prepare_fold on shape {fold.shape}")

    start = time.time()
    sc.pp.pca(fold, n_comps=50)
    log_progress(f"{fold_name}: PCA complete in {time.time() - start:.2f}s")

    for n_pcs in STANDARD_UMAP_PCS:
        neighbors_key = f"neighbors_pcs{n_pcs}"
        umap_key = f"X_umap_pcs{n_pcs}"
        start = time.time()
        sc.pp.neighbors(
            fold,
            use_rep="X_pca",
            n_pcs=n_pcs,
            key_added=neighbors_key,
        )
        sc.tl.umap(fold, neighbors_key=neighbors_key)
        fold.obsm[umap_key] = fold.obsm["X_umap"].copy()
        log_progress(f"{fold_name}: neighbors+UMAP for {n_pcs} PCs complete in {time.time() - start:.2f}s")

    start = time.time()
    log_progress(f"{fold_name}: starting BiPCA fit for MP rank")
    bipca_rank = compute_bipca_rank(fold, seed=SEED)
    fold.uns["bipca_mp_rank"] = bipca_rank
    log_progress(f"{fold_name}: BiPCA fit complete in {time.time() - start:.2f}s with mp_rank={bipca_rank}")

    start = time.time()
    sc.pp.neighbors(
        fold,
        use_rep="X_pca",
        n_pcs=bipca_rank,
        key_added="neighbors_bipca_rank",
    )
    sc.tl.umap(fold, neighbors_key="neighbors_bipca_rank")
    fold.obsm["X_umap_bipca_umap2d"] = fold.obsm["X_umap"].copy()
    log_progress(f"{fold_name}: BiPCA-rank neighbors+UMAP complete in {time.time() - start:.2f}s")

    fold.write(INTERMEDIATE_DIR / f"{fold_name}_prepared_with_embeddings.h5ad")
    log_progress(f"{fold_name}: prepared fold written to intermediate cache; total prepare time {time.time() - fold_start:.2f}s")
    return fold


def locat_dataframe(locat_result_dict):
    df = pd.DataFrame({gene: res._asdict() for gene, res in locat_result_dict.items()}).T
    df.index.name = "gene"
    return df.sort_values("pval", ascending=True)


w_transform = lambda x: np.clip(x, 0.0, np.inf)


def locat_coordinates_for_setting(fold, setting):
    if isinstance(setting, int):
        return fold.obsm["X_pca"].astype(np.float64)[:, :setting]
    if setting == BIPCA_SETTING:
        return fold.obsm["X_umap_bipca_umap2d"].astype(np.float64)
    raise ValueError(f"Unknown setting: {setting}")


def run_locat_on_fold(fold, setting, locat_seed, n_bootstrap_inits=50, max_freq=0.9):
    reset_seeds(locat_seed)
    model = LOCAT(
        fold,
        locat_coordinates_for_setting(fold, setting),
        20,
        show_progress=True,
        n_bootstrap_inits=n_bootstrap_inits,
    )
    result = model.gmm_scan(
        genes=list(fold.var_names),
        weights_transform=w_transform,
        rc_lambda_values=np.linspace(1.0, 2.0, 8),
        include_depletion_scan=True,
        max_freq=max_freq,
    )
    return locat_dataframe(result)


def ranking_frame(df):
    ranked = df.reset_index().rename(columns={"index": "gene"}).copy()
    ranked["rank"] = np.arange(1, len(ranked) + 1)
    return ranked.set_index("gene")


def compare_rankings(df_a, df_b, top_n, alpha=0.05):
    rank_a = ranking_frame(df_a)
    rank_b = ranking_frame(df_b)
    top_a = rank_a.index[:top_n]
    top_b = rank_b.index[:top_n]
    genes = pd.Index(top_a).union(pd.Index(top_b))
    rho, rho_p = spearmanr(rank_a.loc[genes, "rank"], rank_b.loc[genes, "rank"])
    sig_a = set(df_a.index[df_a["pval"] < alpha])
    sig_b = set(df_b.index[df_b["pval"] < alpha])
    union_sig = sig_a | sig_b
    inter_sig = sig_a & sig_b
    return {
        "top_n_union": top_n,
        "n_genes_in_union": len(genes),
        "spearman_rho": rho,
        "spearman_pval": rho_p,
        "sig_a": len(sig_a),
        "sig_b": len(sig_b),
        "intersection": len(inter_sig),
        "union": len(union_sig),
        "jaccard": (len(inter_sig) / len(union_sig)) if union_sig else np.nan,
    }


def compare_rankings_multi(df_a, df_b, label_a, label_b, compare_type, top_ns=(200, 500), alpha=0.05):
    rows = []
    for top_n in top_ns:
        row = compare_rankings(df_a, df_b, top_n=top_n, alpha=alpha)
        row.update(
            {
                "compare_type": compare_type,
                "label_a": label_a,
                "label_b": label_b,
            }
        )
        rows.append(row)
    return rows


def aggregate_metric(df, group_cols, metric_cols):
    grouped = (
        df.groupby(group_cols, dropna=False)[metric_cols]
        .agg(["mean", "std", "min", "max", "count"])
        .reset_index()
    )
    grouped.columns = [
        "__".join([str(part) for part in col if part != ""]).rstrip("_")
        if isinstance(col, tuple) else col
        for col in grouped.columns
    ]
    return grouped


def plot_metric_bars(metric_df, title, metric_col, hue_col, x_col, ylabel):
    plt.figure(figsize=(10, 4.5))
    sns.barplot(data=metric_df, x=x_col, y=metric_col, hue=hue_col, errorbar=None)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def plot_fold_umaps(fold, fold_name):
    panel_settings = [50, 4, 8, 12, BIPCA_SETTING]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.ravel()
    for ax, setting in zip(axes, panel_settings):
        emb = fold.obsm[setting_embedding_key(setting)]
        colors = fold.obs["cell_type"].astype("category").cat.codes
        ax.scatter(emb[:, 0], emb[:, 1], c=colors, s=8, cmap="tab20", linewidths=0, alpha=0.9)
        if setting == BIPCA_SETTING:
            ax.set_title(f"{fold_name}: UMAP from BiPCA rank={fold.uns['bipca_mp_rank']}")
        else:
            ax.set_title(f"{fold_name}: UMAP from {setting} PCs")
        ax.set_xticks([])
        ax.set_yticks([])
    axes[-1].axis("off")
    plt.tight_layout()
    plt.show()


def plot_intersection_genes_across_folds(prepared_folds, results_by_fold_pc_seed, setting_use, seed_use, genes, top_n=20, save_path=None):
    genes = list(genes)[:top_n]
    fig, axes = plt.subplots(len(genes), 2, figsize=(8, 2.8 * len(genes)), squeeze=False)
    for row_i, gene in enumerate(genes):
        for col_i, fold_name in enumerate(["fold_A", "fold_B"]):
            ax = axes[row_i, col_i]
            ad = prepared_folds[fold_name]
            emb = ad.obsm[setting_embedding_key(setting_use)]
            expr = to_dense_1d(ad[:, [gene]].X)
            ax.scatter(emb[:, 0], emb[:, 1], s=5, c="#d9d9d9", linewidths=0, alpha=0.35)
            mask = expr > 0
            if mask.any():
                ax.scatter(
                    emb[mask, 0],
                    emb[mask, 1],
                    s=8,
                    c=expr[mask],
                    cmap="viridis",
                    linewidths=0,
                    alpha=0.95,
                )
            rank_df = results_by_fold_pc_seed[(fold_name, setting_use, seed_use)]
            title = f"{fold_name} | {gene} | {setting_label(setting_use)} | p={rank_df.loc[gene, 'pval']:.2e}"
            ax.set_title(title, fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def plot_foldA_top_genes_on_foldB(prepared_folds, results_by_fold_pc_seed, setting_use, train_seed, top_n=10, save_path=None):
    rank_a = results_by_fold_pc_seed[("fold_A", setting_use, train_seed)]
    rank_b = results_by_fold_pc_seed[("fold_B", setting_use, train_seed)]
    genes = list(rank_a.index[:top_n])
    ad_b = prepared_folds["fold_B"]
    emb = ad_b.obsm[setting_embedding_key(setting_use)]

    fig, axes = plt.subplots(top_n, 1, figsize=(5.8, 2.4 * top_n), squeeze=False)
    axes = axes.ravel()
    for i, gene in enumerate(genes):
        ax = axes[i]
        expr = to_dense_1d(ad_b[:, [gene]].X)
        ax.scatter(emb[:, 0], emb[:, 1], s=5, c="#d9d9d9", linewidths=0, alpha=0.35)
        mask = expr > 0
        if mask.any():
            ax.scatter(
                emb[mask, 0],
                emb[mask, 1],
                s=9,
                c=expr[mask],
                cmap="viridis",
                linewidths=0,
                alpha=0.95,
            )
        p_a = rank_a.loc[gene, "pval"]
        p_b = rank_b.loc[gene, "pval"]
        rank_b_pos = int(rank_b.index.get_loc(gene)) + 1
        ax.set_title(
            f"Fold A top gene on Fold B | {setting_label(setting_use)} | {gene} | "
            f"A p={p_a:.2e}, B p={p_b:.2e}, B rank={rank_b_pos}",
            fontsize=9,
        )
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    return genes


In [ ]:
fold_dict = build_two_disjoint_folds(stim, frac=1/3, random_state=SEED)

for fold_name, fold in fold_dict.items():
    print(fold_name, fold.shape)
    print(fold.obs["cell_type"].value_counts().head(), end="\n\n")


In [ ]:
shared_genes_df = select_shared_genes(
    fold_dict,
    n_genes=2000,
    min_frac=0.01,
    max_frac=0.80,
)

shared_genes_df.head()


In [ ]:
print("Shared eligible genes retained:", len(shared_genes_df))
shared_genes_df["avg_variance"].describe()


In [ ]:

prepared_folds = {}
for fold_name in ["fold_A", "fold_B"]:
    prepared_folds[fold_name] = prepare_fold(fold_dict[fold_name], shared_genes_df, fold_name)

{k: v.shape for k, v in prepared_folds.items()}


In [ ]:

for fold_name, fold in prepared_folds.items():
    print(f"{fold_name} BiPCA MP rank:", fold.uns["bipca_mp_rank"])
    plot_fold_umaps(fold, fold_name)


## LOCAT runs


In [ ]:

results_by_fold_pc_seed = {}
total_models = len([f for f in ["fold_A", "fold_B"]]) * len(SETTING_LIST) * len(LOCAT_SEEDS)
completed_models = 0

for fold_name in ["fold_A", "fold_B"]:
    for setting in SETTING_LIST:
        for locat_seed in LOCAT_SEEDS:
            model_start = time.time()
            log_progress(
                f"Running LOCAT {completed_models + 1}/{total_models}: {fold_name}, setting={setting_label(setting)}, seed={locat_seed}"
            )
            result_df = run_locat_on_fold(
                prepared_folds[fold_name],
                setting=setting,
                locat_seed=locat_seed,
                n_bootstrap_inits=50,
                max_freq=0.9,
            )
            results_by_fold_pc_seed[(fold_name, setting, locat_seed)] = result_df
            result_path = INTERMEDIATE_DIR / f"{fold_name}_{setting_slug(setting)}_seed{locat_seed}_locat_results.csv"
            result_df.to_csv(result_path)
            completed_models += 1
            log_progress(
                f"Completed LOCAT {completed_models}/{total_models}: {fold_name}, setting={setting_label(setting)}, seed={locat_seed} in {time.time() - model_start:.2f}s; n_sig_p005={(result_df['pval'] < 0.05).sum()}"
            )


In [ ]:

result_summary_rows = []
for (fold_name, setting, locat_seed), df in results_by_fold_pc_seed.items():
    result_summary_rows.append(
        {
            "fold": fold_name,
            "setting": setting,
            "setting_label": setting_label(setting),
            "setting_slug": setting_slug(setting),
            "seed": locat_seed,
            "n_sig_p005": int((df["pval"] < 0.05).sum()),
            "top_gene": df.index[0],
            "top_pval": float(df.iloc[0]["pval"]),
        }
    )

result_summary_df = pd.DataFrame(result_summary_rows).sort_values(["fold", "setting_label", "seed"])
result_summary_df


## Stability analyses


In [ ]:

comparison_rows = []

for setting in SETTING_LIST:
    slug = setting_slug(setting)
    for locat_seed in LOCAT_SEEDS:
        df_a = results_by_fold_pc_seed[("fold_A", setting, locat_seed)]
        df_b = results_by_fold_pc_seed[("fold_B", setting, locat_seed)]
        comparison_rows.extend(
            compare_rankings_multi(
                df_a,
                df_b,
                label_a=f"fold_A_{slug}_seed{locat_seed}",
                label_b=f"fold_B_{slug}_seed{locat_seed}",
                compare_type=f"across_folds_same_setting_{slug}",
            )
        )

for fold_name in ["fold_A", "fold_B"]:
    for setting in SETTING_LIST:
        slug = setting_slug(setting)
        for seed_a, seed_b in combinations(LOCAT_SEEDS, 2):
            df_a = results_by_fold_pc_seed[(fold_name, setting, seed_a)]
            df_b = results_by_fold_pc_seed[(fold_name, setting, seed_b)]
            comparison_rows.extend(
                compare_rankings_multi(
                    df_a,
                    df_b,
                    label_a=f"{fold_name}_{slug}_seed{seed_a}",
                    label_b=f"{fold_name}_{slug}_seed{seed_b}",
                    compare_type=f"within_fold_seed_variability_{fold_name}_{slug}",
                )
            )

for fold_name in ["fold_A", "fold_B"]:
    for locat_seed in LOCAT_SEEDS:
        for setting_a, setting_b in combinations(SETTING_LIST, 2):
            slug_a = setting_slug(setting_a)
            slug_b = setting_slug(setting_b)
            df_a = results_by_fold_pc_seed[(fold_name, setting_a, locat_seed)]
            df_b = results_by_fold_pc_seed[(fold_name, setting_b, locat_seed)]
            comparison_rows.extend(
                compare_rankings_multi(
                    df_a,
                    df_b,
                    label_a=f"{fold_name}_{slug_a}_seed{locat_seed}",
                    label_b=f"{fold_name}_{slug_b}_seed{locat_seed}",
                    compare_type=f"within_fold_setting_sensitivity_{fold_name}_{slug_a}__vs__{slug_b}",
                )
            )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.head()


In [ ]:

across_fold_df = comparison_df[comparison_df["compare_type"].str.startswith("across_folds_same_setting")].copy()
within_seed_df = comparison_df[comparison_df["compare_type"].str.startswith("within_fold_seed_variability")].copy()
setting_sensitivity_df = comparison_df[comparison_df["compare_type"].str.startswith("within_fold_setting_sensitivity")].copy()
pc_sensitivity_df = setting_sensitivity_df.copy()

print(across_fold_df.shape, within_seed_df.shape, setting_sensitivity_df.shape)


In [ ]:
across_fold_summary = aggregate_metric(
    across_fold_df,
    group_cols=["compare_type", "top_n_union"],
    metric_cols=["spearman_rho", "jaccard", "intersection", "union"],
)
across_fold_summary


In [ ]:
within_seed_summary = aggregate_metric(
    within_seed_df,
    group_cols=["compare_type", "top_n_union"],
    metric_cols=["spearman_rho", "jaccard", "intersection", "union"],
)
within_seed_summary


In [ ]:

pc_sensitivity_summary = aggregate_metric(
    setting_sensitivity_df,
    group_cols=["compare_type", "top_n_union"],
    metric_cols=["spearman_rho", "jaccard", "intersection", "union"],
)
pc_sensitivity_summary


In [ ]:

across_fold_plot_rows = []
for setting in SETTING_LIST:
    slug = setting_slug(setting)
    subset = across_fold_df[across_fold_df["compare_type"] == f"across_folds_same_setting_{slug}"].copy()
    subset["setting"] = setting_label(setting)
    across_fold_plot_rows.append(subset)
across_fold_plot_df = pd.concat(across_fold_plot_rows, ignore_index=True)

within_seed_plot_rows = []
for setting in SETTING_LIST:
    slug = setting_slug(setting)
    subset = within_seed_df[within_seed_df["compare_type"].str.endswith(slug)].copy()
    subset["setting"] = setting_label(setting)
    within_seed_plot_rows.append(subset)
within_seed_plot_df = pd.concat(within_seed_plot_rows, ignore_index=True)

setting_order = [setting_label(x) for x in SETTING_LIST]
across_fold_plot_df["setting"] = pd.Categorical(across_fold_plot_df["setting"], categories=setting_order, ordered=True)
within_seed_plot_df["setting"] = pd.Categorical(within_seed_plot_df["setting"], categories=setting_order, ordered=True)

seed_baseline = (
    within_seed_plot_df.groupby(["setting", "top_n_union"])[["spearman_rho", "jaccard"]]
    .mean()
    .reset_index()
    .rename(columns={"spearman_rho": "seed_mean_rho", "jaccard": "seed_mean_jaccard"})
)
seed_baseline


In [ ]:

fold_vs_seed = across_fold_plot_df.merge(seed_baseline, on=["setting", "top_n_union"], how="left")
fold_vs_seed[["compare_type", "top_n_union", "setting", "spearman_rho", "seed_mean_rho", "jaccard", "seed_mean_jaccard"]]


In [ ]:

plt.figure(figsize=(9.5, 4.5))
sns.barplot(data=across_fold_plot_df, x="setting", y="spearman_rho", hue="top_n_union", errorbar=None)
for idx, setting_name in enumerate(setting_order):
    subset = seed_baseline[seed_baseline["setting"] == setting_name]
    for _, row in subset.iterrows():
        plt.hlines(
            row["seed_mean_rho"],
            xmin=idx - 0.38,
            xmax=idx + 0.38,
            colors="black" if row["top_n_union"] == 200 else "gray",
            linestyles="--",
            linewidth=1.3,
        )
plt.title("Across-fold rank correlation by LOCAT setting")
plt.ylabel("Spearman rho")
plt.xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(9.5, 4.5))
sns.barplot(data=across_fold_plot_df, x="setting", y="jaccard", hue="top_n_union", errorbar=None)
for idx, setting_name in enumerate(setting_order):
    subset = seed_baseline[seed_baseline["setting"] == setting_name]
    for _, row in subset.iterrows():
        plt.hlines(
            row["seed_mean_jaccard"],
            xmin=idx - 0.38,
            xmax=idx + 0.38,
            colors="black" if row["top_n_union"] == 200 else "gray",
            linestyles="--",
            linewidth=1.3,
        )
plt.title("Across-fold significant-gene overlap by LOCAT setting")
plt.ylabel("Jaccard overlap")
plt.xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## Shared top genes across folds


In [ ]:

shared_sig_intersections = {}
for setting in SETTING_LIST:
    for locat_seed in LOCAT_SEEDS:
        sig_a = set(results_by_fold_pc_seed[("fold_A", setting, locat_seed)].index[
            results_by_fold_pc_seed[("fold_A", setting, locat_seed)]["pval"] < 0.05
        ])
        sig_b = set(results_by_fold_pc_seed[("fold_B", setting, locat_seed)].index[
            results_by_fold_pc_seed[("fold_B", setting, locat_seed)]["pval"] < 0.05
        ])
        shared_sig_intersections[(setting, locat_seed)] = sorted(sig_a & sig_b)

{k: len(v) for k, v in shared_sig_intersections.items()}


In [ ]:

plot_setting = 8
plot_seed = LOCAT_SEEDS[0]

genes_a = list(results_by_fold_pc_seed[("fold_A", plot_setting, plot_seed)].index[:40])
genes_b = list(results_by_fold_pc_seed[("fold_B", plot_setting, plot_seed)].index[:40])
overlapping_top_genes = [g for g in genes_a if g in genes_b]
overlapping_top_genes[:20]


In [ ]:

plot_intersection_genes_across_folds(
    prepared_folds=prepared_folds,
    results_by_fold_pc_seed=results_by_fold_pc_seed,
    setting_use=plot_setting,
    seed_use=plot_seed,
    genes=overlapping_top_genes,
    top_n=20,
)


## Held-out visualization of Fold A top genes on Fold B

For each PC setting, take the top 10 genes ranked by LOCAT in Fold A (using a fixed seed) and visualize those same genes on the matching Fold B embedding.

In [ ]:

FIGURE_DIR = Path("stim_twothirds_cv_figures")
FIGURE_DIR.mkdir(exist_ok=True, parents=True)

heldout_plot_seed = LOCAT_SEEDS[0]
heldout_genes_by_setting = {}

for setting in SETTING_LIST:
    save_path = FIGURE_DIR / f"foldA_top10_on_foldB_{setting_slug(setting)}_seed{heldout_plot_seed}.png"
    print(f"Saving held-out validation plot: {save_path}")
    heldout_genes_by_setting[setting] = plot_foldA_top_genes_on_foldB(
        prepared_folds=prepared_folds,
        results_by_fold_pc_seed=results_by_fold_pc_seed,
        setting_use=setting,
        train_seed=heldout_plot_seed,
        top_n=10,
        save_path=save_path,
    )


In [ ]:

heldout_summary_rows = []
for setting, genes in heldout_genes_by_setting.items():
    rank_a = results_by_fold_pc_seed[("fold_A", setting, heldout_plot_seed)]
    rank_b = results_by_fold_pc_seed[("fold_B", setting, heldout_plot_seed)]
    for rank_a_pos, gene in enumerate(genes, start=1):
        heldout_summary_rows.append({
            "setting": setting,
            "setting_label": setting_label(setting),
            "seed": heldout_plot_seed,
            "gene": gene,
            "fold_A_rank": rank_a_pos,
            "fold_A_pval": float(rank_a.loc[gene, "pval"]),
            "fold_B_rank": int(rank_b.index.get_loc(gene)) + 1,
            "fold_B_pval": float(rank_b.loc[gene, "pval"]),
            "fold_B_significant_p005": bool(rank_b.loc[gene, "pval"] < 0.05),
        })

heldout_summary_df = pd.DataFrame(heldout_summary_rows)
heldout_summary_df


## AUROC recovery summaries and table-ready outputs

This section computes recovery-style AUROC summaries directly from the notebook results for the same three comparison settings used in the supplementary tables: across folds, within-fold across seeds, and within-fold across PCA dimensionalities.

In [ ]:

from scipy.stats import t
from sklearn.metrics import roc_auc_score

AUROC_EPS = 1e-12
SETTING_PAIR_LIST = list(combinations(SETTING_LIST, 2))


def ci_half_width(values):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) <= 1:
        return np.nan
    return t.ppf(0.975, len(vals) - 1) * vals.std(ddof=1) / np.sqrt(len(vals))


def logp_series(df, eps=AUROC_EPS):
    return -np.log10(df["pval"].astype(float).clip(lower=0) + eps)


def format_pm(mean_val, ci_half):
    return f"{mean_val:.3f} $\pm$ {ci_half:.3f}"


def recovery_auroc(y_ref, score_series):
    if y_ref.nunique() < 2:
        return np.nan
    return roc_auc_score(y_ref.astype(int), score_series.astype(float))


def recovery_auroc_values_across_folds(results_by_fold_pc_seed, setting, seeds):
    values = []
    for seed_a in seeds:
        for seed_b in seeds:
            key_a = ("fold_A", setting, seed_a)
            key_b = ("fold_B", setting, seed_b)
            df_a = results_by_fold_pc_seed[key_a]
            df_b = results_by_fold_pc_seed[key_b]
            for ref_df, score_df in [(df_a, df_b), (df_b, df_a)]:
                y_ref = ref_df["pval"] < 0.05
                score = logp_series(score_df).reindex(ref_df.index)
                values.append(recovery_auroc(y_ref, score))
    return values


def recovery_auroc_values_within_seed(results_by_fold_pc_seed, setting, seeds):
    values = []
    for fold_name in ["fold_A", "fold_B"]:
        for i, seed_a in enumerate(seeds):
            for seed_b in seeds[i + 1:]:
                key_a = (fold_name, setting, seed_a)
                key_b = (fold_name, setting, seed_b)
                df_a = results_by_fold_pc_seed[key_a]
                df_b = results_by_fold_pc_seed[key_b]
                for ref_df, score_df in [(df_a, df_b), (df_b, df_a)]:
                    y_ref = ref_df["pval"] < 0.05
                    score = logp_series(score_df).reindex(ref_df.index)
                    values.append(recovery_auroc(y_ref, score))
    return values


def recovery_auroc_values_setting_pair(results_by_fold_pc_seed, setting_a, setting_b, seeds):
    values = []
    for fold_name in ["fold_A", "fold_B"]:
        for locat_seed in seeds:
            key_a = (fold_name, setting_a, locat_seed)
            key_b = (fold_name, setting_b, locat_seed)
            df_a = results_by_fold_pc_seed[key_a]
            df_b = results_by_fold_pc_seed[key_b]
            for ref_df, score_df in [(df_a, df_b), (df_b, df_a)]:
                y_ref = ref_df["pval"] < 0.05
                score = logp_series(score_df).reindex(ref_df.index)
                values.append(recovery_auroc(y_ref, score))
    return values


def summarize_auroc_across_folds(results_by_fold_pc_seed, settings, seeds):
    rows = []
    for setting in settings:
        values = recovery_auroc_values_across_folds(results_by_fold_pc_seed, setting, seeds)
        rows.append({
            "setting": setting,
            "setting_label": setting_label(setting),
            "auroc_mean": float(np.nanmean(values)),
            "auroc_ci_half": float(ci_half_width(values)),
            "n": len(values),
        })
    return pd.DataFrame(rows)


def summarize_auroc_within_seed(results_by_fold_pc_seed, settings, seeds):
    rows = []
    for setting in settings:
        values = recovery_auroc_values_within_seed(results_by_fold_pc_seed, setting, seeds)
        rows.append({
            "setting": setting,
            "setting_label": setting_label(setting),
            "auroc_mean": float(np.nanmean(values)),
            "auroc_ci_half": float(ci_half_width(values)),
            "n": len(values),
        })
    return pd.DataFrame(rows)


def summarize_auroc_setting_sensitivity(results_by_fold_pc_seed, setting_pairs, seeds):
    rows = []
    for setting_a, setting_b in setting_pairs:
        values = recovery_auroc_values_setting_pair(results_by_fold_pc_seed, setting_a, setting_b, seeds)
        rows.append({
            "pair": f"{setting_short_label(setting_a)} vs {setting_short_label(setting_b)}",
            "setting_a": setting_a,
            "setting_b": setting_b,
            "auroc_mean": float(np.nanmean(values)),
            "auroc_ci_half": float(ci_half_width(values)),
            "n": len(values),
        })
    return pd.DataFrame(rows)


def metric_pm_from_values(values):
    vals = np.asarray(values, dtype=float)
    return format_pm(float(np.nanmean(vals)), float(ci_half_width(vals)))


def build_across_fold_table(across_fold_df, across_fold_auroc_df):
    columns = ["metric", "pcsfour", "pcseight", "pcstwelve", "bipca_umap2d"]
    rows = []
    row = {"metric": "AUROC (all genes; $-\log_{10} p$)"}
    for setting, col in zip(SETTING_LIST, columns[1:]):
        sub = across_fold_auroc_df.loc[across_fold_auroc_df["setting"] == setting].iloc[0]
        row[col] = format_pm(sub["auroc_mean"], sub["auroc_ci_half"])
    rows.append(row)

    for metric_name, top_n in [("Spearman rho (top 200)", 200), ("Spearman rho (top 500)", 500)]:
        row = {"metric": metric_name}
        for setting, col in zip(SETTING_LIST, columns[1:]):
            slug = setting_slug(setting)
            vals = across_fold_df[(across_fold_df["compare_type"] == f"across_folds_same_setting_{slug}") & (across_fold_df["top_n_union"] == top_n)]["spearman_rho"]
            row[col] = metric_pm_from_values(vals)
        rows.append(row)

    row = {"metric": "Jaccard overlap (significant genes)"}
    for setting, col in zip(SETTING_LIST, columns[1:]):
        slug = setting_slug(setting)
        vals = across_fold_df[(across_fold_df["compare_type"] == f"across_folds_same_setting_{slug}") & (across_fold_df["top_n_union"] == 200)]["jaccard"]
        row[col] = metric_pm_from_values(vals)
    rows.append(row)
    return pd.DataFrame(rows, columns=columns)


def build_within_seed_table(within_seed_df, within_seed_auroc_df):
    columns = ["metric", "pcsfour", "pcseight", "pcstwelve", "bipca_umap2d"]
    rows = []
    row = {"metric": "AUROC (all genes; $-\log_{10} p$)"}
    for setting, col in zip(SETTING_LIST, columns[1:]):
        sub = within_seed_auroc_df.loc[within_seed_auroc_df["setting"] == setting].iloc[0]
        row[col] = format_pm(sub["auroc_mean"], sub["auroc_ci_half"])
    rows.append(row)

    for metric_name, top_n in [("Spearman rho (top 200)", 200), ("Spearman rho (top 500)", 500)]:
        row = {"metric": metric_name}
        for setting, col in zip(SETTING_LIST, columns[1:]):
            slug = setting_slug(setting)
            vals = within_seed_df[within_seed_df["compare_type"].str.endswith(slug) & (within_seed_df["top_n_union"] == top_n)]["spearman_rho"]
            row[col] = metric_pm_from_values(vals)
        rows.append(row)

    row = {"metric": "Jaccard overlap (significant genes)"}
    for setting, col in zip(SETTING_LIST, columns[1:]):
        slug = setting_slug(setting)
        vals = within_seed_df[within_seed_df["compare_type"].str.endswith(slug) & (within_seed_df["top_n_union"] == 200)]["jaccard"]
        row[col] = metric_pm_from_values(vals)
    rows.append(row)
    return pd.DataFrame(rows, columns=columns)


def build_setting_sensitivity_table(setting_sensitivity_df, setting_sensitivity_auroc_df):
    pair_columns = [
        (setting_a, setting_b, f"{setting_slug(setting_a)}__vs__{setting_slug(setting_b)}")
        for setting_a, setting_b in SETTING_PAIR_LIST
    ]
    columns = ["metric"] + [col for _, _, col in pair_columns]
    rows = []

    row = {"metric": "AUROC (all genes; $-\log_{10} p$)"}
    for setting_a, setting_b, col in pair_columns:
        pair_label = f"{setting_short_label(setting_a)} vs {setting_short_label(setting_b)}"
        sub = setting_sensitivity_auroc_df.loc[setting_sensitivity_auroc_df["pair"] == pair_label].iloc[0]
        row[col] = format_pm(sub["auroc_mean"], sub["auroc_ci_half"])
    rows.append(row)

    for metric_name, top_n in [("Spearman rho (top 200)", 200), ("Spearman rho (top 500)", 500)]:
        row = {"metric": metric_name}
        for setting_a, setting_b, col in pair_columns:
            vals = setting_sensitivity_df[
                setting_sensitivity_df["compare_type"].str.endswith(f"{setting_slug(setting_a)}__vs__{setting_slug(setting_b)}")
                & (setting_sensitivity_df["top_n_union"] == top_n)
            ]["spearman_rho"]
            row[col] = metric_pm_from_values(vals)
        rows.append(row)

    row = {"metric": "Jaccard overlap (significant genes)"}
    for setting_a, setting_b, col in pair_columns:
        vals = setting_sensitivity_df[
            setting_sensitivity_df["compare_type"].str.endswith(f"{setting_slug(setting_a)}__vs__{setting_slug(setting_b)}")
            & (setting_sensitivity_df["top_n_union"] == 200)
        ]["jaccard"]
        row[col] = metric_pm_from_values(vals)
    rows.append(row)
    return pd.DataFrame(rows, columns=columns)


In [ ]:

across_fold_auroc_df = summarize_auroc_across_folds(results_by_fold_pc_seed, SETTING_LIST, LOCAT_SEEDS)
within_seed_auroc_df = summarize_auroc_within_seed(results_by_fold_pc_seed, SETTING_LIST, LOCAT_SEEDS)
setting_sensitivity_auroc_df = summarize_auroc_setting_sensitivity(results_by_fold_pc_seed, SETTING_PAIR_LIST, LOCAT_SEEDS)
pc_sensitivity_auroc_df = setting_sensitivity_auroc_df.copy()

table_across_fold_with_auroc_df = build_across_fold_table(across_fold_df, across_fold_auroc_df)
table_within_seed_with_auroc_df = build_within_seed_table(within_seed_df, within_seed_auroc_df)
table_pc_sensitivity_with_auroc_df = build_setting_sensitivity_table(setting_sensitivity_df, setting_sensitivity_auroc_df)

print("Across-fold AUROC summary")
display(across_fold_auroc_df)
print("Within-seed AUROC summary")
display(within_seed_auroc_df)
print("Across-setting AUROC summary")
display(setting_sensitivity_auroc_df)

print("Across-fold table-ready summary")
display(table_across_fold_with_auroc_df)
print("Within-seed table-ready summary")
display(table_within_seed_with_auroc_df)
print("Across-setting table-ready summary")
display(table_pc_sensitivity_with_auroc_df)


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

axes[0].bar(across_fold_auroc_df["setting_label"], across_fold_auroc_df["auroc_mean"], color="#4C78A8")
axes[0].errorbar(
    np.arange(len(across_fold_auroc_df)),
    across_fold_auroc_df["auroc_mean"],
    yerr=across_fold_auroc_df["auroc_ci_half"],
    fmt="none",
    ecolor="black",
    capsize=4,
    linewidth=1.2,
)
axes[0].set_title("Across-fold AUROC")
axes[0].set_xlabel("")
axes[0].set_ylabel("Recovery AUROC")
axes[0].set_ylim(0.85, 1.01)
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(within_seed_auroc_df["setting_label"], within_seed_auroc_df["auroc_mean"], color="#59A14F")
axes[1].errorbar(
    np.arange(len(within_seed_auroc_df)),
    within_seed_auroc_df["auroc_mean"],
    yerr=within_seed_auroc_df["auroc_ci_half"],
    fmt="none",
    ecolor="black",
    capsize=4,
    linewidth=1.2,
)
axes[1].set_title("Within-seed AUROC")
axes[1].set_xlabel("")
axes[1].set_ylim(0.85, 1.01)
axes[1].tick_params(axis="x", rotation=15)

axes[2].bar(setting_sensitivity_auroc_df["pair"], setting_sensitivity_auroc_df["auroc_mean"], color="#E15759")
axes[2].errorbar(
    np.arange(len(setting_sensitivity_auroc_df)),
    setting_sensitivity_auroc_df["auroc_mean"],
    yerr=setting_sensitivity_auroc_df["auroc_ci_half"],
    fmt="none",
    ecolor="black",
    capsize=4,
    linewidth=1.2,
)
axes[2].set_title("Across-setting AUROC")
axes[2].set_xlabel("")
axes[2].set_ylim(0.85, 1.01)
axes[2].tick_params(axis="x", rotation=25)

plt.tight_layout()
auroc_plot_path = FIGURE_DIR / "stim_twothirds_cv_auroc_summary_panels.png"
fig.savefig(auroc_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved AUROC summary panel: {auroc_plot_path}")
plt.show()


## Save tables

In [ ]:

OUTPUT_DIR = Path("<LOCAL_ROOT>/Locat/data")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

shared_genes_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_shared_2000_genes_locat01.csv", index=False)
result_summary_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_result_summary_locat01.csv", index=False)
comparison_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_all_comparisons_locat01.csv", index=False)
across_fold_summary.to_csv(OUTPUT_DIR / "stim_twothirds_cv_across_fold_summary_locat01.csv", index=False)
within_seed_summary.to_csv(OUTPUT_DIR / "stim_twothirds_cv_within_seed_summary_locat01.csv", index=False)
pc_sensitivity_summary.to_csv(OUTPUT_DIR / "stim_twothirds_cv_pc_sensitivity_summary_locat01.csv", index=False)

for (fold_name, setting, locat_seed), df in results_by_fold_pc_seed.items():
    df.to_csv(
        OUTPUT_DIR / f"{fold_name}_stim_locat01_{setting_slug(setting)}_seed{locat_seed}_twothirds_cv.csv"
    )
heldout_summary_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_foldA_top10_on_foldB_summary_locat01.csv", index=False)
across_fold_auroc_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_across_fold_auroc_summary_locat01.csv", index=False)
within_seed_auroc_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_within_seed_auroc_summary_locat01.csv", index=False)
setting_sensitivity_auroc_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_setting_sensitivity_auroc_summary_locat01.csv", index=False)
pc_sensitivity_auroc_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_pc_sensitivity_auroc_summary_locat01.csv", index=False)
table_across_fold_with_auroc_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_table_across_fold_with_auroc_locat01.csv", index=False)
table_within_seed_with_auroc_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_table_within_seed_with_auroc_locat01.csv", index=False)
table_pc_sensitivity_with_auroc_df.to_csv(OUTPUT_DIR / "stim_twothirds_cv_table_pc_sensitivity_with_auroc_locat01.csv", index=False)
